##**Producer-Consumer Pipeline**

In [1]:
import threading
import queue
import time
import random

# 1. Create an infinite capacity queue (maxsize=0 or empty means unlimited)
sensor_queue = queue.Queue()

# Share a stop flag to shut down loops cleanly later
stop_pipeline = threading.Event()

def producer_sensor():
    """
    PRODUCER: Simulates a sensor reading data every 0.1 seconds
    and pushing it onto the queue conveyor belt.
    """
    print(f"[{threading.current_thread().name}] Sensor streaming started.")
    reading_id = 1

    while not stop_pipeline.is_set():
        # Simulate data creation (e.g., a mock temperature value)
        mock_data = {
            "id": reading_id,
            "val": round(random.uniform(20.0, 25.0), 2)
        }

        # Pushing data onto the queue
        sensor_queue.put(mock_data)
        print(f"[{threading.current_thread().name}] Sent packet #{reading_id}: {mock_data['val']}°C")

        reading_id += 1
        time.sleep(0.1)  # Produce every 0.1s

    print(f"[{threading.current_thread().name}] Producer safely stopped.")


def consumer_processor():
    """
    CONSUMER: Pulls readings off the conveyor belt and processes them
    at a matching speed of 0.1 seconds.
    """
    print(f"[{threading.current_thread().name}] Core data processing loop active.")

    while not stop_pipeline.is_set() or not sensor_queue.empty():
        try:
            # Try to grab data with a short timeout so it doesn't get stuck forever on shutdown
            data_packet = sensor_queue.get(timeout=0.2)

            # Simulate analyzing/processing the data
            print(f"    [{threading.current_thread().name}] Caught packet #{data_packet['id']}: Processing data...")
            time.sleep(0.1)  # Process every 0.1s

            # Checkmark: Tell the queue this specific item is done
            sensor_queue.task_done()

        except queue.Empty:
            # Safe exception if queue is temporarily empty during a check
            continue

    print(f"[{threading.current_thread().name}] Consumer safely parked.")

# --- Execution ---
# Define and instantiate threads
prod_thread = threading.Thread(target=producer_sensor, name="Sensor_Producer")
cons_thread = threading.Thread(target=consumer_processor, name="IK_Consumer")

# Spin up loops
prod_thread.start()
cons_thread.start()

# Let the pipeline stream steadily for 1.5 seconds
time.sleep(1.5)

# Trigger clean shutdown sequence
print("\n[Main] Stopping simulation...")
stop_pipeline.set()

# Wait for threads to finalize current work items
prod_thread.join()
cons_thread.join()

print("[Main] Execution complete. Queue is completely empty.")

[Sensor_Producer] Sensor streaming started.
[Sensor_Producer] Sent packet #1: 21.97°C
[IK_Consumer] Core data processing loop active.
    [IK_Consumer] Caught packet #1: Processing data...
[Sensor_Producer] Sent packet #2: 24.36°C
    [IK_Consumer] Caught packet #2: Processing data...
[Sensor_Producer] Sent packet #3: 20.9°C
    [IK_Consumer] Caught packet #3: Processing data...
[Sensor_Producer] Sent packet #4: 21.75°C
    [IK_Consumer] Caught packet #4: Processing data...
[Sensor_Producer] Sent packet #5: 23.72°C
    [IK_Consumer] Caught packet #5: Processing data...
[Sensor_Producer] Sent packet #6: 23.36°C
    [IK_Consumer] Caught packet #6: Processing data...
[Sensor_Producer] Sent packet #7: 22.05°C
    [IK_Consumer] Caught packet #7: Processing data...
[Sensor_Producer] Sent packet #8: 24.76°C
    [IK_Consumer] Caught packet #8: Processing data...
[Sensor_Producer] Sent packet #9: 23.58°C
    [IK_Consumer] Caught packet #9: Processing data...
[Sensor_Producer] Sent packet #10: 2